# Gemma 2B: Base Model vs RLHF Instruct Model

This notebook loads both versions of Gemma 2B and lets you compare their responses to the same prompts.

- **Base model** (`google/gemma-2b`): pre-trained on text, just predicts the next token
- **Instruct model** (`google/gemma-2b-it`): fine-tuned with RLHF to follow instructions

> **Before running:** Make sure you have accepted the Gemma license at https://huggingface.co/google/gemma-2b

## Step 1: Install dependencies


In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00


## Step 2: Log in to Hugging Face

Get your token from https://huggingface.co/settings/tokens. Use a Read token.

In [ ]:
from huggingface_hub import login

login(token="hf_enter_your_token_from_huggingface")

## Step 3: Load the Base Model

This is Gemma 2B **before** RLHF. It only knows how to complete text.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

print("Loading base model...")

base_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
base_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b",
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Base model loaded.")

Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Base model loaded.


## Step 4: Load the Instruct Model

This is Gemma 2B **after** RLHF. It has been fine-tuned to follow instructions.

In [ ]:
print("Loading instruct model...")

instruct_tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b-it")
instruct_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2b-it",
    device_map="auto",
    torch_dtype=torch.float16,
)

print("Instruct model loaded.")

Loading instruct model...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Instruct model loaded.


## Step 5: Helper functions

In [ ]:
def run_base_model(prompt, max_new_tokens=200):
    """Run the base model — it will try to complete your text."""
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=base_tokenizer.eos_token_id
    )
    return base_tokenizer.decode(outputs[0], skip_special_tokens=True)


def run_instruct_model(prompt, max_new_tokens=200):
    """Run the instruct model — it will try to answer your question."""
    inputs = instruct_tokenizer(prompt, return_tensors="pt").to(instruct_model.device)
    outputs = instruct_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=instruct_tokenizer.eos_token_id
    )
    return instruct_tokenizer.decode(outputs[0], skip_special_tokens=True)


def compare(prompt):
    """Run the same prompt through both models and print the results."""
    print(f"PROMPT: {prompt}")
    print("-" * 60)
    print("BASE MODEL (pre-RLHF):")
    print(run_base_model(prompt))
    print()
    print("INSTRUCT MODEL (post-RLHF):")
    print(run_instruct_model(prompt))
    print("=" * 60)

## Step 6: Compare the models

Run the same prompts through both. Notice how differently they respond.


In [7]:
# The base model will likely continue this as a travel blog or list
# The instruct model will answer it like an assistant
compare("What are good things to do in London?")

PROMPT: What are good things to do in London?
------------------------------------------------------------
BASE MODEL (pre-RLHF):
What are good things to do in London? A lot of people visit London because of the many historical buildings and landmarks. Other people visit London because of its many museums, galleries, and galleries. Some people visit London because of its nightlife and entertainment. Those are just a few things people can do in London.

There are many things to do in London. The top three things are to visit the Tower of London, go to the London Eye, and go to the London Dungeon. The Tower of London is the most popular tourist attraction in London. It has many buildings and rooms that were used for different purposes over the years. Some of the buildings and rooms were used for storing the royal jewels, while others were used for holding prisoners.

The London Eye is the tallest Ferris wheel in the world. It is located on the south bank of the Thames. It has 32 capsules